In [ ]:
from pelmesha.pspectra import DataProc_1d, peaks_prop_infunc,msalign,smoothing, Configs
from pelmesha.pfeats import Pgrouping_KD, Getrefpeaks, Pgrouping_KD_table
import pandas as pd
import xarray as xr
import numpy as np
from multiprocessing import cpu_count
from itertools import pairwise, product
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
def ESI_peaks(path, configs, signal_distortions, scan_filter = None, free_cores = 2, draw = True, grouping_peaks = True, **Pgrouping_KD_configs): #TODO: Не универсальная функция!!!! Пока только локальная
    """
    path: path to netcdf file
    configs: configs for DataProc_1d and peaks_prop_infunc
    signal_distortions: list of tuples with signal distortions
    scan_filter: list of scan filters
    free_cores: number of free cores
    draw: draw plots
    Pgrouping_KD_configs: configs for Pgrouping_KD
    """

    ds = xr.open_dataset(path, engine='netcdf4')
    mz = ds.variables['mass_values'].values

    intens = ds.variables['intensity_values'].values
    scan_dots_sequence = np.append(ds['scan_index'].values, len(mz))
    scan_dots_slice = [slice(*x,1) for x in pairwise(scan_dots_sequence)]
    scan_sequence = np.array(range(len(ds['scan_filters'])))

    DataProc_configs = configs["DataProc_configs"]
    PeakPicking_configs = configs["peaks_configs"]
    peaklists = {}

    mz_min = min(mz)
    mz_max = max(mz)
    if scan_filter is None:
        scan_filter = np.unique(ds['scan_filters'])
    elif isinstance(scan_filter, (int, np.int32, np.int64)):
        scan_filter = [scan_filter]
    filtered_scans = [[i, scan_sequence[np.array(ds['scan_filters'].values == i, dtype=bool)]] for i in np.unique(ds['scan_filters']) if i in scan_filter]
    
    for i, scans in filtered_scans:
        data = [0]*len(scans)
        for n, scan in enumerate(scans):
            slc = scan_dots_slice[scan]
            if i == 2 or i == 3:
                artefacts_bool = np.zeros(len(mz[slc]), dtype=bool)
                loc_mz = mz[slc]
                for distortion in signal_distortions:
                    artefacts_bool = artefacts_bool | ((loc_mz > distortion[0]) & (loc_mz < distortion[1]))
                loc_ints = intens[slc]
                loc_ints[artefacts_bool] = 0
            else:
                loc_mz = mz[slc]
                loc_ints = intens[slc]

            if configs['resample_to_dots']:
                resampled_mz = np.linspace(mz_min, mz_max,configs['resample_to_dots'])
                loc_ints = np.interp(resampled_mz, loc_mz, loc_ints)
                loc_mz = resampled_mz

        
            data[n] = (loc_mz, loc_ints, scan)
        par_args = list(product(data,[DataProc_configs],[PeakPicking_configs]))
        peaklists[i] = Parallel(n_jobs= cpu_count()-free_cores)(delayed(ESI_int2proc2peaklist_parbatched)(*data) for data in par_args)
        peaklists[i] = pd.DataFrame(np.vstack(peaklists[i]), columns = PeakPicking_configs['headers'])
        if grouping_peaks:
            peaklists[i] = Pgrouping_KD(peaklists[i], sample = f'Scan filter', roi = f"{i}", draw = draw,**Pgrouping_KD_configs)
            if draw:
                if isinstance(draw,(list,tuple)):
                    iter_draw = draw[1]
                else:
                    iter_draw = 1
                Leg_1 = []
                Leg_2 = []
                ax = plt.gca().twinx()
                for i in range(iter_draw):
                    if i == 0:
                        rand_spec_1 = Pgrouping_KD_table.rand_spec_1
                        rand_spec_2 = Pgrouping_KD_table.rand_spec_2
                    else:
                        if Pgrouping_KD_table.rand_spec_1:
                            rand_spec_1 = [np.random.choice(filtered_scans[0][1])]
                        else:
                            rand_spec_1 = None
                        rand_spec_2 = np.random.choice(filtered_scans[0][1])
                    if rand_spec_1:
                        
                        rand_spec = rand_spec_1[-1]
                        if not isinstance(rand_spec, int):
                            rand_spec = int(rand_spec)
                        

                        plt.figure(num=plt.get_fignums()[-2])
                        slc = scan_dots_slice[rand_spec]
                        x = np.array(mz[slc])
                        y = np.array(intens[slc])
                        dots_distance = np.quantile(np.diff(x),0.33)

                        DataProc_configs['smoothing_configs']['smooth_window'](dots_distance) 
                        DataProc_configs['msalign_configs']['shift_range'](dots_distance)
                        DataProc_configs['baseliner'](x)
                        mz_draw_borders = plt.xlim()
                        dots_bord_spec = (x>=mz_draw_borders[0]) & (x<=mz_draw_borders[1])
                        
                        ax.plot(x[dots_bord_spec],y[dots_bord_spec],alpha=0.85)
                        if configs['resample_to_dots']:
                            x = np.linspace(x.min(), x.max(), configs['resample_to_dots'])
                            dots_distance = np.quantile(np.diff(x),0.33)
            
                            DataProc_configs['smoothing_configs']['smooth_window'](dots_distance) 
                            DataProc_configs['msalign_configs']['shift_range'](dots_distance)
                            DataProc_configs['baseliner'](x)
                        proc_y = DataProc_1d(y, x,**DataProc_configs)
                        dots_bord_spec = (x>=mz_draw_borders[0]) & (x<=mz_draw_borders[1])
                        plt.plot(x[dots_bord_spec], proc_y[dots_bord_spec],alpha=0.85)
                        plt.gcf().tight_layout()
                        plt.ylabel("Intensity")
                        Leg_1+=[f"Raw mass spectrum N {rand_spec}", f"Processed mass spectrum N {rand_spec}"]
                        
                        plt.legend(Leg_1,loc='upper left')
                                
                    if not isinstance(rand_spec_2, int):
                        rand_spec_2 = int(rand_spec_2)
                    plt.figure(num=plt.get_fignums()[-1])
                    slc = scan_dots_slice[rand_spec_2]
                    x = np.array(mz[slc])
                    y = np.array(intens[slc])
                    dots_distance = np.quantile(np.diff(x),0.33)

                    DataProc_configs['smoothing_configs']['smooth_window'](dots_distance) 
                    DataProc_configs['msalign_configs']['shift_range'](dots_distance)
                    DataProc_configs['baseliner'](x)
                    mz_draw_borders = plt.xlim()
                    dots_bord_spec = (x>=mz_draw_borders[0]) & (x<=mz_draw_borders[1])
                    ax.plot(x[dots_bord_spec],y[dots_bord_spec],alpha=0.85)
                    if configs['resample_to_dots']:
                        x = np.linspace(x.min(), x.max(), configs['resample_to_dots'])
                        dots_distance = np.quantile(np.diff(x),0.33)
        
                        DataProc_configs['smoothing_configs']['smooth_window'](dots_distance) 
                        DataProc_configs['msalign_configs']['shift_range'](dots_distance)
                        DataProc_configs['baseliner'](x)
                    proc_y = DataProc_1d(y, x,**DataProc_configs)
                    dots_bord_spec = (x>=mz_draw_borders[0]) & (x<=mz_draw_borders[1])
                    plt.plot(x[dots_bord_spec], proc_y[dots_bord_spec],alpha=0.85)

                    plt.gcf().tight_layout()
                    plt.ylabel("Intensity")
                    Leg_2 += [f"Raw mass spectrum N {rand_spec_2}", f"Processed mass spectrum N {rand_spec_2}"]
                    plt.legend(Leg_2, loc='upper left')


    return peaklists
            

def ESI_int2proc2peaklist_parbatched(data, DataProc_configs, PeakPicking_configs):
    

    mz, ints, scan = data
    dots_distance = np.quantile(np.diff(mz),0.33)
    
    DataProc_configs['smoothing_configs']['smooth_window'](dots_distance) 
    DataProc_configs['msalign_configs']['shift_range'](dots_distance)
    DataProc_configs['baseliner'](mz)
    proc_ints = DataProc_1d(ints, mz,**DataProc_configs)
    peaklist = peaks_prop_infunc(mz, proc_ints, np.where(np.diff(proc_ints) != 0)[0], len(mz),
                                scan, **PeakPicking_configs)

    return peaklist

In [ ]:
# Наблюдаемые диапазоны наводок орбитрепа (список неполный и недостаточно расширенный). Интенсивности в этих диапазонах зануляются в процессе обработки.
signal_distortions = [(102.09, 103.86),
                      (107.79, 108.61), 
                      (127.5, 127.85), 
                     (155.7, 156.6), 
                     (168.5, 170.8),
                     (188.9, 189.15),
                     (202.00, 202.55),
                     (203.13, 203.65), 
                     (210.30, 210.60), 
                     (220.10, 221.02), 
                     (229.36, 231.53), 
                     (241.92, 242.56), 
                     (285.83, 287.11), 
                     (398.18, 401.15), 
                     (472.29, 472.89), 
                     (508.34, 512.61), 
                     (620.00, 622.81), 
                     (643.60, 645.03), 
                     (674.0, 676.17), 
                     (677.37, 683.2), 
                     (906.92, 908.83), 
                     (510.0, 511.2), 
                     (622.5, 627.5), 
                     (643.0, 644.5), 
                     (676.0, 679.5)]
path = r'C:\Job_and_Literature\24.cdf'


# Фильтрация шумовых пиков Orbitrap
На основе [данной работы](https://pubs.acs.org/doi/10.1021/ac403278t)

In [ ]:
# Создаём итоговый пик-лист после выравнивания и без выравнивания (просто для сравнения)
peaklists = {}

for i in [2,3]:
    if i == 2 or i == 3: # Настройки для обработки Орбитреп данных делаем без какой-либо фильтрации пиков, фильтр шумовых пиков сделаем далее. Отстутсвие фильтра на этом этапе - ключевой момент, так как дальнейший фильтр основывается на статистике
        oversegmentationfilter = 0
        SNR_threshold = 0 
        Shift_range = 0.25
        resample_to_dots = None #150000
        CountF = 0

    configs = Configs([DataProc_1d,peaks_prop_infunc,msalign,smoothing],
                      align_peaks = None,
                      align_pweights = None,
                      shift_range = Shift_range,
                      resample_to_dots = resample_to_dots,
                  smooth_algo = 'GA', 
                  smooth_window = 0.5, 
                #   baseline_algo = 'asls', 
                  SNR_threshold = SNR_threshold, 
                  oversegmentationfilter = oversegmentationfilter)

    peaklists.update(ESI_peaks(path, configs,signal_distortions,i, draw_borders = 3, account_mzscale = False, CountF = CountF, return_pkY = True, draw = (True,3)))

In [ ]:
from KDEpy import FFTKDE
from scipy.signal import argrelextrema
p = {}
for i in [2,3]:
    p= np.log10(peaklists[i]['Intensity']).copy() #преобразуем интенсивности в десятичные логарифмы
    #Построим функцию плотности
    ints = np.array(p.loc[p>np.log10(0.01)]) #Для ускорения рассчётов - исключим пики с интенсивностями явно шумовыми
    X_plot = np.arange(ints.min()-0.001,ints.max()+0.001,0.00001) #Построим равномерную сетку для построении функции плотности вероятности десятичного логарифма
    Y_kde = FFTKDE(bw ="scott" ).fit(ints)(X_plot) #Определяем плотность вероятности десятичного логарифма интенсивностей с использованием функции построения KDE с применением фурье-преобразования (пакет KDEpy)
    peaks = argrelextrema(Y_kde,np.less) # Находим индексы всех локальных минимумов, для отрисовки соответствующие им значения интенсивностей на графиках
    intensity = 10**X_plot[peaks] # Преобразуем в интенсивности обратно для отрисовки
    plt.figure(figsize=(25,5))
    plt.plot(X_plot,Y_kde)
    plt.scatter(X_plot[peaks],Y_kde[peaks],100,'k','|')
    plt.xlabel("Intensity. log10 scale")
    plt.ylabel("Probabilty density")
    plt.legend(['Intensity KDE function','local minima'])
    plt.xlim((ints.min(),ints.max()))
    # Добавим подписи к минимумам
    for x,y,s in zip(list(X_plot[peaks]),(Y_kde[peaks]),[f'{x:.2f}' for x in intensity]):
        plt.text(x,y+0.05,s,rotation=90,verticalalignment='bottom',horizontalalignment='center')

        

# Выравнивание по пиклисту

In [ ]:
# Настраиваем параметры обработки
configs = Configs([DataProc_1d,peaks_prop_infunc,msalign,smoothing],
                  smooth_algo = 'GA', 
                  smooth_window = 0.5, 
                  baseline_algo = 'asls', 
                  SNR_threshold = 9, 
                  oversegmentationfilter = 0.05)

# Настройки для функции Getrefpeaks (получение референснного пик-листа с весами относительно которого производить  выравнивание)

step = 50
num_peaks_per_step = 5 
min_occurence = 0.5
return_weight = True

# Получаем пик-лист общий
peaklists = ESI_peaks(path, configs, signal_distortions, account_mzscale = False)
# Получаем пик-лист референсный
for i in peaklists.keys():
    peaklists[i] = Getrefpeaks(peaklists[i], 
                                step, 
                                num_peaks_per_step, 
                                min_occurence, 
                                return_weight,
                                CountF = 100,
                                draw_borders = 5,
                                return_pkY=True)


In [ ]:
# Создаём итоговый пик-лист после выравнивания и без выравнивания (просто для сравнения)
aln_peaklists = {}
noaln_peaklists = {}
for i in peaklists.keys():
    if i == 2 or i == 3: # Настройки для обработки Орбитреп данных делаем без какой-либо фильтрации пиков, фильтр шумовых пиков сделаем далее. Отстутсвие фильтра на этом этапе - ключевой момент, так как дальнейший фильтр основывается на статистике
        oversegmentationfilter = 0.05
        SNR_threshold = 0 
        Shift_range = 0.25
        resample_to_dots = None #150000
        CountF = 0
    else:
        Shift_range = 0.45
        SNR_threshold = 5
        oversegmentationfilter = 0.1
        resample_to_dots = None
        CountF = 25
    
    configs = Configs([DataProc_1d,peaks_prop_infunc,msalign,smoothing],
                      align_peaks = peaklists[i][0], 
                      align_pweights = peaklists[i][1],
                      shift_range = Shift_range,
                      resample_to_dots = resample_to_dots,
                  smooth_algo = 'GA', 
                  smooth_window = 0.5, 
                  baseline_algo = 'asls', 
                  SNR_threshold = SNR_threshold, 
                  oversegmentationfilter = oversegmentationfilter)
    print(configs['SNR_threshold'])
    aln_peaklists.update(ESI_peaks(path, configs,signal_distortions,i, draw_borders = 5, CountF = CountF, return_pkY = True))
    configs['align_peaks'] = None
    configs['align_pweights'] = None
    noaln_peaklists.update(ESI_peaks(path, configs,signal_distortions,i, draw_borders = 5, CountF = CountF, return_pkY = True))
    

Предполагаем, что фильтр по интеснивности находится на уровне 1626. Отфильтровываем пики, которые меньше 1626.
Возможно шумовые пики даже выше на самом деле, так как данные с наводкой могут вносить искажения.

In [ ]:
# Фильтруем шумовые пики и сразу же перегруппируем пики.
account_mzscale = True
bwc = 1
aln_peaklists_gr = {}
noaln_peaklists_gr = {}
for i in [2,3]:
    print(f"aligned data. Scan filter {i}")
    aln_peaklists_gr[i] = Pgrouping_KD(aln_peaklists[i].query("Intensity > 1626"), CountF = 10,bwc = bwc, account_mzscale = account_mzscale,return_pkY=True, sample= "scan filter", roi = f"{i} aligned data")
    print(f"No aligned data. Scan filter {i}")
    noaln_peaklists_gr[i] = Pgrouping_KD(noaln_peaklists[i].query("Intensity > 1626"), CountF = 10, bwc = bwc, account_mzscale = account_mzscale,return_pkY=True, sample= "scan filter", roi = f"{i} no aligned data")


In [ ]:
# Создаём итоговый пик-лист после выравнивания и без выравнивания (просто для сравнения)
aln_peaklists = {}
noaln_peaklists = {}
for i in peaklists.keys():
    if i == 2 or i == 3: # Настройки для обработки Орбитреп данных делаем без какой-либо фильтрации пиков, фильтр шумовых пиков сделаем далее. Отстутсвие фильтра на этом этапе - ключевой момент, так как дальнейший фильтр основывается на статистике
        oversegmentationfilter = 0.05
        SNR_threshold = 5
        # heightfilter = 260000 # фильтр пиков по высоте, выставляем найденное значение
        heightfilter = None
        Shift_range = 0.25
        resample_to_dots = 200000
        CountF = 10
    else:
        Shift_range = 0.45
        SNR_threshold = 3
        oversegmentationfilter = 0.1
        resample_to_dots = None
        heightfilter = None
        CountF = 10
    
    configs = Configs([DataProc_1d,peaks_prop_infunc,msalign,smoothing],
                      align_peaks = peaklists[i][0], 
                      align_pweights = peaklists[i][1],
                      shift_range = Shift_range,
                      resample_to_dots = resample_to_dots,
                  smooth_algo = 'GA', 
                  smooth_window = 0.5, 
                  baseline_algo = 'asls', 
                  SNR_threshold = SNR_threshold,
                  heightfilter = heightfilter,
                  oversegmentationfilter = oversegmentationfilter)
    print(configs['SNR_threshold'])
    aln_peaklists.update(ESI_peaks(path, configs,signal_distortions,i, draw_borders = 5, CountF = CountF, return_pkY = True, account_mzscale = False))
    configs['align_peaks'] = None
    configs['align_pweights'] = None
    noaln_peaklists.update(ESI_peaks(path, configs,signal_distortions,i, draw_borders = 5, CountF = CountF, return_pkY = True, account_mzscale = False))

Отрисовка результата по сканам в определённом диапазоне

In [ ]:
import matplotlib.pyplot as plt

xlim= [1500,1600] # диапазон отрисовки
for reg in aln_peaklists.keys():
    plt.figure(figsize=(20,7.5))
    mz = aln_peaklists[reg]['mz']
    mz_noaln = noaln_peaklists[reg]['mz']
    mz_feat = aln_peaklists_gr[reg]['Peak']

    mz_bool = (mz>xlim[0]) & (mz<xlim[1])
    mz_noaln_bool = (mz_noaln>xlim[0]) & (mz_noaln<xlim[1])
    mz_feat_bool = (mz_feat>xlim[0]) & (mz_feat<xlim[1])

    mz = mz[mz_bool]
    mz_noaln = mz_noaln[mz_noaln_bool]
    mz_feat = mz_feat[mz_feat_bool]

    spectra_ind = aln_peaklists[reg]['spectra_ind'][mz_bool]
    spectra_ind_noaln = noaln_peaklists[reg]['spectra_ind'][mz_noaln_bool]
    spectra_ind_feat = aln_peaklists_gr[reg]['spectra_ind'][mz_feat_bool]

    intensity = aln_peaklists[reg]['Intensity'][mz_bool]
    intensity_noaln = noaln_peaklists[reg]['Intensity'][mz_noaln_bool]
    intensity_feat = intensity

    
    plt.scatter(mz_noaln, spectra_ind_noaln, s = 2, label='Not aligned', color ='g')
    plt.scatter(mz, spectra_ind, s = 4, label='Aligned', alpha=0.75, color='k')
    plt.scatter(mz_feat, spectra_ind_feat, s = 0.75, label='Features', color='r')
    # plt.vlines(mz_feat.unique(), ymin=0, ymax=max(spectra_ind_feat), label='Features', color='r')
    # print(spectra_ind)
    if not spectra_ind.empty:
        plt.ylim(spectra_ind.min(),spectra_ind.max())
    plt.ylabel('Spectra index')
    plt.xlabel('m/z')
    plt.title(f"scan filter {reg}")
    plt.legend(['Aligned', 'Not aligned', 'Features'], markerscale=5)
    plt.xlim(xlim)
    plt.show()


# Вариант выравнивания по усреднённому масс спектру (не доделано)

In [ ]:

mz_min = mz.max() 
mz_max = mz.min()
resample_dots_distance = 0.01

# 3) Если равномерность точек отсутствует, то сделать ресемплинг mz и intens
# 4) Проссумировать интенсивности (разумнее, чем делать индивидуальный пик-пикинг и смотреть по частоте встречаемости: 1) Это LC - если часто встречается - это мусор)
# 5) Построить график
# 6) Найти пики
# 7) Создать референсный список

for i in np.unique(ds['scan_filters']):
    # filtered_scans = scan_dots_slice[np.array(ds['scan_filters'].values == i, dtype=bool)]
    filtered_scans = scan_sequence[np.array(ds['scan_filters'].values == i, dtype=bool)]
    mz_min = mz.max() 
    mz_max = mz.min()
    for scan in filtered_scans:
        mz_min = min(mz_min, *mz[scan_dots_slice[scan]])
        mz_max = max(mz_max, *mz[scan_dots_slice[scan]])
    dots =np.int64( (mz_max - mz_min) / resample_dots_distance)
    print(mz_min, mz_max,dots)

    
    resampled_mz = np.linspace(mz_min, mz_max, dots)
    # Подготовка всех спектров сразу
    all_mz, all_intens = zip(*((mz[scan_dots_slice[scan]], intens[scan_dots_slice[scan]]) for scan in filtered_scans))

    # Векторизованная интерполяция через numpy
    data_int = np.array(tuple(
        np.interp(resampled_mz, mz, intens, 
                left=intens[0], right=intens[-1])
        for mz, intens in zip(all_mz, all_intens)
    ))
    plt.figure(figsize=(20, 5))
    
    plt.plot(resampled_mz, np.sum(data_int, axis=0), label=i)
    plt.xlim(780, 790)

In [ ]:
from pelmesha.pspectra import DataProc_1d, peaks_prop_infunc,msalign,smoothing, Configs
from pelmesha.pfeats import Pgrouping_KD, Getrefpeaks
import pandas as pd


data_int_noaln = {}
data_mz_draw = {}
data_mz_draw_noaln = {}
mz_min = min(mz)
mz_max = max(mz)

resample_to_dots = 150000

for i in np.unique(ds['scan_filters']):
    data_mz_draw[i] = {}
    
    configs = Configs([DataProc_1d,peaks_prop_infunc,msalign,smoothing],
                      align_peaks = peaklists[i][0], 
                      align_pweights = peaklists[i][1], 
                      shift_range = 0.45, 
                      smooth_algo = 'GA', 
                      smooth_window = 0.5,
                      baseline_algo = 'asls', 
                      SNR_threshold = 3, 
                      oversegmentationfilter = 0.05)
    DataProc_configs = configs["DataProc_configs"]
    PeakPicking_configs = configs["peaks_configs"]
    peakl = {}
    filtered_scans = scan_sequence[np.array(ds['scan_filters'].values == i, dtype=bool)]
    data_int = {}
    for n, scan in enumerate(filtered_scans):
        slc = scan_dots_slice[scan]         
        data_mz, data_int[n] = mz[slc], intens[slc]
        if i == 2 or i == 3:
            artefacts_bool = np.zeros(len(data_mz), dtype=bool)
            for distortion in signal_distortions:
                artefacts_bool = artefacts_bool | ((data_mz > distortion[0]) & (data_mz < distortion[1]))
            # artefacts_bool = ((data_mz > 676) & (data_mz < 680)) | ((data_mz > 155) & (data_mz < 157)) | ((data_mz > 168) & (data_mz < 171)) | ((data_mz > 624) & (data_mz < 627)) | ((data_mz > 510) & (data_mz < 512))
            # data_mz, data_int = data_mz[~artefacts_bool], data_int[~artefacts_bool]
            data_int[n][artefacts_bool] = 0
            configs['resample_to_dots'] = resample_to_dots
            old_mz = data_mz
            data_mz = np.linspace(mz_min, mz_max,configs['resample_to_dots'])
            data_mz_draw[i] = data_mz
            data_int[n] = np.interp(np.linspace(mz_min, mz_max,configs['resample_to_dots']), old_mz, data_int[n])
        else:
            data_mz_draw[i][n] = data_mz
        dots_distance = np.median(np.diff(data_mz))
        DataProc_configs['smoothing_configs']['smooth_window'](dots_distance) 
        DataProc_configs['msalign_configs']['shift_range'](dots_distance)
        DataProc_configs['baseliner'](data_mz)

        data_int[n] = DataProc_1d(data_int[n],data_mz,**DataProc_configs)
        peakl[n] = peaks_prop_infunc(data_mz, data_int[n], np.where(np.diff(data_int[n]) != 0)[0], len(data_mz),
                                    scan, **PeakPicking_configs)
    aligned_peaklists[i] = pd.DataFrame(np.vstack(tuple(peakl.values())), columns = PeakPicking_configs['headers'])
    aligned_peaklists[i] = Pgrouping_KD(aligned_peaklists[i], 
                                        CountF= 10, 
                                        # KD_bandwidth='mz_discret',
                                        draw_borders= 3,
                                        return_pkY=True)
    xlim = plt.xlim()
    ax = plt.gca().twinx()
    n = np.random.randint(0, len(filtered_scans))
    slc = scan_dots_slice[filtered_scans[n]]
    mz_plot = mz[slc]
    intens_plot = intens[slc]

    mz_bool = (mz_plot > xlim[0]) & (mz_plot < xlim[1])
    if i == 2 or i == 3:
        mz_draw_bool = (data_mz_draw[i] > xlim[0]) & (data_mz_draw[i] < xlim[1])
        ax.plot(data_mz_draw[i][mz_draw_bool], data_int[n][mz_draw_bool], color = 'green', label = f"Processed mass spectrum N{scan}")
    else:
        mz_draw_bool = (data_mz_draw[i][n] > xlim[0]) & (data_mz_draw[i][n] < xlim[1])
        ax.plot(data_mz_draw[i][n][mz_draw_bool], data_int[n][mz_draw_bool], color = 'green', label = f"Processed mass spectrum N{scan}")
    ax.plot(mz_plot[mz_bool], intens_plot[mz_bool], color = 'red', label = f"Raw mass spectrum N{scan}")
    ax.set_xlabel("m/z")
    ax.set_ylabel("Intensity")
    plt.legend()
    plt.show()

    configs['align_peaks'] = None
    configs['align_pweights'] = None
    
    DataProc_configs = configs["DataProc_configs"]
    PeakPicking_configs = configs["peaks_configs"]
    peakl = {}
    data_mz_draw[i] = {}
    data_int = {}
    for n, scan in enumerate(filtered_scans):
        
        slc = scan_dots_slice[scan]         
        data_mz, data_int[n] = mz[slc], intens[slc]
        if i == 2 or i == 3:
            artefacts_bool = np.zeros(len(data_mz), dtype=bool)
            for distortion in signal_distortions:
                artefacts_bool = artefacts_bool | ((data_mz > distortion[0]) & (data_mz < distortion[1]))
            # artefacts_bool = ((data_mz > 676) & (data_mz < 680)) | ((data_mz > 155) & (data_mz < 157)) | ((data_mz > 168) & (data_mz < 171)) | ((data_mz > 624) & (data_mz < 627)) | ((data_mz > 510) & (data_mz < 512))
            # data_mz, data_int = data_mz[~artefacts_bool], data_int[~artefacts_bool]
            data_int[n][artefacts_bool] = 0
            configs['resample_to_dots'] = resample_to_dots
            old_mz = data_mz
            data_mz = np.linspace(mz_min, mz_max,configs['resample_to_dots'])
            data_mz_draw[i] = data_mz
            data_int[n] = np.interp(np.linspace(mz_min, mz_max,configs['resample_to_dots']), old_mz, data_int[n])
        else:
            data_mz_draw[i][n] = data_mz
        dots_distance = np.median(np.diff(data_mz))
        DataProc_configs['smoothing_configs']['smooth_window'](dots_distance) 
        DataProc_configs['msalign_configs']['shift_range'](dots_distance)
        DataProc_configs['baseliner'](data_mz)

        data_int[n] = DataProc_1d(data_int[n],data_mz,**DataProc_configs)
        peakl[n] = peaks_prop_infunc(data_mz, data_int[n], np.where(np.diff(data_int[n]) != 0)[0], len(data_mz),
                                    scan, **PeakPicking_configs)
    noaln_peaklists[i] = pd.DataFrame(np.vstack(tuple(peakl.values())), columns = PeakPicking_configs['headers'])
    noaln_peaklists[i] = Pgrouping_KD(noaln_peaklists[i], 
                                        CountF= 10, 
                                        # KD_bandwidth='mz_discret',
                                        draw_borders= 3,
                                        return_pkY=True)
    xlim = plt.xlim()
    ax = plt.gca().twinx()


    n = np.random.randint(0, len(filtered_scans))
    slc = scan_dots_slice[filtered_scans[n]]
    mz_plot = mz[slc]
    intens_plot = intens[slc]
    mz_bool = (mz_plot > xlim[0]) & (mz_plot < xlim[1])
    if i == 2 or i == 3:
        mz_draw_bool = (data_mz_draw[i] > xlim[0]) & (data_mz_draw[i] < xlim[1])
        ax.plot(data_mz_draw[i][mz_draw_bool], data_int[n][mz_draw_bool], color = 'green', label = f"Processed mass spectrum N{scan}")
    else:
        mz_draw_bool = (data_mz_draw[i][n] > xlim[0]) & (data_mz_draw[i][n] < xlim[1])
        ax.plot(data_mz_draw[i][n][mz_draw_bool], data_int[n][mz_draw_bool], color = 'green', label = f"Processed mass spectrum N{scan}")
    ax.plot(mz_plot[mz_bool], intens_plot[mz_bool], color = 'red', label = f"Raw mass spectrum N{scan}")
    ax.set_ylabel("Intensity")
    plt.legend()
    plt.show()
